In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download(
    "bhavikjikadara/dog-and-cat-classification-dataset"
)

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/bhavikjikadara/dog-and-cat-classification-dataset


In [1]:
import os

import matplotlib.pyplot as plt


class GraphBase:
    def __init__(self, folder_path):
        self.folder_path = os.path.join(folder_path, "graphs")
        if not os.path.exists(self.folder_path):
            os.makedirs(self.folder_path)
        plt.style.use("dark_background")

    def build_graph(self):
        raise NotImplementedError("Subclasses should implement this method")

In [2]:
import pandas as pd
import seaborn as sns


class ImageLabelClassification(GraphBase):
    def __init__(
        self,
        labels,
        label2class_mapping,
        invalid_labels,
        folder_path,
        title="",
        file_name="",
    ):
        super().__init__(folder_path)
        label2class_mapping = label2class_mapping
        self.valid_df = pd.DataFrame({"labels": labels})
        self.valid_df["labels"] = self.valid_df["labels"].map(label2class_mapping)
        self.total_df = None
        self.invalid_df = None
        self.title = title
        self.file_name = file_name
        if invalid_labels is not None and len(invalid_labels) > 0:
            self.invalid_df = pd.DataFrame({"labels": invalid_labels})
            self.invalid_df["labels"] = self.invalid_df["labels"].map(
                label2class_mapping
            )
            self.total_df = pd.concat(
                [self.valid_df, self.invalid_df], axis=0, ignore_index=True
            )
            self.length = [0, 0]
            self.length[0] = len(self.valid_df["labels"]) / len(
                self.total_df["labels"]
            )
            self.length[1] = len(self.invalid_df["labels"]) / len(
                self.total_df["labels"]
            )
        else:
            self.total_df = self.valid_df
            self.length = [1, 0]

    def build_graph(self):
        if self.invalid_df is not None:
            fig, ax = plt.subplots(1, 4, figsize=(16, 5))
        else:
            fig, ax = plt.subplots(1, 3, figsize=(12, 5))

        fig.suptitle(self.title, fontsize=20)
        sns.countplot(data=self.total_df, x="labels", ax=ax[0])
        ax[0].set_title("Total images classes distribution")
        ax[0].set_xlabel("Classes")
        ax[0].set_ylabel("Count")
        ax[1].pie(
            [float(self.length[0] * 100), float(100 * self.length[1])],
            labels=["valid images", "invalid images"],
            autopct="%1.1f%%",
            colors=["#021D25", "#ADD8E6"],
        )
        ax[1].set_title("Valid vs Invalid percentage")
        sns.countplot(data=self.valid_df, x="labels", ax=ax[2])
        ax[2].set_title("Valid images classes distribution")
        ax[2].set_xlabel("Classes")
        ax[2].set_ylabel("Count")
        if self.invalid_df is not None:
            sns.countplot(data=self.invalid_df, x="labels", ax=ax[3])
            ax[3].set_title("Invalid images classes distribution")
            ax[3].set_xlabel("Classes")
            ax[3].set_ylabel("Count")
        plt.tight_layout()
        plt.savefig(os.path.join(self.folder_path, f"{self.file_name}.png"))
        plt.close()

In [3]:
import numpy as np


class DisplaySampleImagesClassification(GraphBase):
    def __init__(
        self,
        paths,
        labels,
        label2class_mapping,
        folder_path,
        n_sample=4,
        title="",
        file_name="",
    ):
        super().__init__(folder_path)
        self.label2class_mapping = label2class_mapping
        self.title = title
        self.file_name = file_name
        self.n_sample = n_sample
        classes = list(self.label2class_mapping.keys())
        self.df = pd.DataFrame({"paths": paths, "labels": labels})
        self.temp_df = {}
        for class_idx in classes:
            class_paths = self.df[self.df["labels"] == class_idx]
            class_df = class_paths.sample(
                n=min(self.n_sample, len(class_paths)), random_state=42
            )
            self.temp_df[label2class_mapping[class_idx]] = class_df["paths"]

    def build_graph(self):
        classes = list(self.temp_df.keys())
        num_rows = len(classes)
        fig, ax = plt.subplots(
            num_rows, self.n_sample, figsize=(4 * self.n_sample, 4 * num_rows)
        )
        fig.suptitle(self.title, fontsize=20)
        ax = np.atleast_2d(ax)
        for row, class_name in enumerate(classes):
            paths = self.temp_df[class_name]
            for col, path in enumerate(paths):
                ax[row][col].axis("off")
                img = plt.imread(path)
                ax[row][col].imshow(img)
                ax[row][col].set_title(class_name)
        plt.tight_layout()
        plt.savefig(os.path.join(self.folder_path, f"{self.file_name}.png"))
        plt.close()

In [4]:
class ClassificationImagesAfterLoader(GraphBase):
    def __init__(
        self,
        images,
        labels,
        label2class_mapping,
        folder_path,
        title="",
        file_name="",
    ):
        super().__init__(folder_path)
        self.images = images
        self.labels = labels
        self.title = title
        self.file_name = file_name
        self.label2class_mapping = label2class_mapping

    def build_graph(self):
        cols = len(self.images)
        fig, ax = plt.subplots(1, cols, figsize=(4 * cols, 4))
        if cols == 1:
            ax = [ax]
        fig.suptitle(self.title, fontsize=20)
        for i, img in enumerate(self.images):
            img = img.permute(1, 2, 0).numpy()
            img = np.clip(img, 0, 1)
            label = self.labels[i].item()
            class_name = self.label2class_mapping[label]
            ax[i].imshow(img)
            ax[i].set_title(class_name)
            ax[i].axis("off")

        plt.tight_layout()
        plt.savefig(os.path.join(self.folder_path, f"{self.file_name}.png"))
        plt.close()

In [5]:
class ReportBase:
    def __init__(self, folderpath):
        self.folderpath = folderpath
        self.start_content = """
        <!DOCTYPE html>
        <html>
        <head>
          <meta charset="UTF-8">
          <meta name="viewport" content="width=device-width, initial-scale=1.0">
          <title>Accelera Report</title>
          <style>
          body {
            font-family: Arial, sans-serif;
            line-height: 1.6;
            background-color: #000000;
            color: #ffffff;
            margin-left:30px
            }
            h1 {
            color: #ffcc00;
            text-align: center;
            }
            h2{
            font-style:italic;
            text-align: center;
            }
            h3{
            color: #ffee00;
            }
            h4 {
            color: #ffcc00; 
            }
            
            table{
            border-collapse: collapse;
            width: 80%;
            }
             th, td {
            border: 1px solid #444;
            padding: 6px;
            text-align: center;
            }
            .metric-container {
            display: grid;
            grid-template-columns: repeat(2, minmax(0, 1fr));
            gap: 20px;
            }
            div{
                margin:40px 10px ;
            }
        </style>
        </head>
        <body>
        <h1>Accelera Report</h1>
        """
        self.end_content = """
        </body>
        </html>
        """

    def show_graphs(self, graphs):
        if graphs is None:
            return
        self.content += "<div>\n"
        self.content += "<h2>Graphs</h2>\n"
        folder_path = os.path.join(graphs["folder_path"], "graphs")
        if not os.path.exists(folder_path):
            self.content += (
                f"<h3>There is no any image in the folder {folder_path}</h3>\n"
            )
        else:
            for image_name in graphs["images_name"]:
                image_file = os.path.join(folder_path, f"{image_name}.png")
                if os.path.exists(image_file):
                    image_file = os.path.join(".", "graphs", f"{image_name}.png")
                    self.content += f"<img src='{image_file}' "
                    self.content += "style='max-width:100%; margin:10px 0;'/>\n"
        self.content += "</div>\n"

    def create_html_file(self, content):
        readme_path = os.path.join(self.folderpath, "report.html")
        with open(readme_path, encoding="utf-8", mode="w") as f:
            f.write(content)

    def execute(self):
        raise NotImplementedError("Subclasses must implement this method")

In [6]:
class ImagePreprocessingReport(ReportBase):
    def __init__(self, folderpath, report_data):
        super().__init__(folderpath)
        self.content = ""
        self.report_data = report_data
        self.data_overview_training = self.report_data["data_overview"].get(
            "training_folder"
        )
        self.data_overview_validation = self.report_data["data_overview"].get(
            "validation_folder"
        )
        self.split_data = self.report_data.get("split_data")
        self.graphs = self.report_data.get("graphs")

    def show_unoreder_list(self, classes, title=None):
        if classes is not None:
            self.content += "<div>\n"
            if title is not None:
                self.content += f"{title}\n"
            self.content += "<ul>\n"
            for class_name in classes:
                self.content += f"<li>{class_name}</li>\n"
            self.content += "</ul>\n"
            self.content += "</div>\n"

    def show_paragraph_section(self, text, title=None):
        self.content += "<div>\n"
        if title is not None:
            self.content += f"{title}\n"
        self.content += f"<p>{text}</p>\n"
        self.content += "</div>\n"

    def show_table_section_from_df(self, dictionary, title=None):
        self.content += "<div>\n"
        if title is not None:
            self.content += f"{title}\n"
        self.content += dictionary.to_html(index=False)
        self.content += "</div>\n"

    def show_folder_overview(self, over_view, folder_type):
        self.content += "<div>\n"
        self.content += f"<h3>{folder_type} Folder Overview</h2>\n"
        self.show_unoreder_list(over_view.get("classes"), "<h4>Classes</h4>")
        if over_view.get("mapping") is not None:
            df = pd.DataFrame(
                over_view["mapping"].items(), columns=["Class", "Label"]
            )
            self.show_table_section_from_df(df, "<h4>Classes 2 Labels Mapping </h4>")
        self.show_paragraph_section(
            f"Count : {over_view['images_len']}", "<h4>Total Images </h4>"
        )
        self.content += "<div>\n"
        self.content += "<h4>Invalid Images</h4>\n"
        self.show_paragraph_section(f"Count : {over_view['invalid_len']}")
        self.show_unoreder_list(over_view["invalid_images"])
        self.content += "</div>\n"
        self.show_table_section_from_df(
            over_view["random_sample"], "<h4>Random Sample</h4>\n"
        )
        self.content += "</div>\n"

    def show_overview(self):
        self.content += "<div>\n"
        self.content += "<h2>Data Overview</h2>\n"
        self.show_folder_overview(self.data_overview_training, "Training")
        if self.data_overview_validation is not None:
            self.show_folder_overview(self.data_overview_training, "Validation")
        self.content += "</div>\n"

    def show_data_split(self):
        if self.split_data is None:
            return
        self.content += "<div>\n"
        self.content += "<h2>Splitting</h2>\n"
        self.show_paragraph_section(
            f"{self.split_data['validation_size']}",
            "<h3>Validation Size</h3>\n",
        )
        self.content += "<div>\n"
        self.content += "<h3>Training After Splitting</h3>\n"
        self.show_paragraph_section(
            f"{self.split_data['training_data_size']}", "<h4>Total Images </h4>"
        )
        self.show_table_section_from_df(
            self.split_data["random_training_sample"],
            "<h4>Random Sample</h4>\n",
        )

        self.content += "</div>\n"
        self.content += "<div>\n"
        self.content += "<h3>Validation After Splitting</h3>\n"
        self.show_paragraph_section(
            f"{self.split_data['validation_data_size']}",
            "<h4>Total Images </h4>",
        )
        self.show_table_section_from_df(
            self.split_data["random_validation_sample"],
            "<h4>Random Sample</h4>\n",
        )
        self.content += "</div>\n"
        self.content += "</div>\n"

    def execute(self):
        self.show_overview()
        self.show_data_split()
        self.show_graphs(self.graphs)
        full_content = self.start_content + self.content + self.end_content
        self.create_html_file(full_content)

In [7]:
import pickle

import torch
from PIL import Image


def check_path_exists(folder_path, filename):
    path = os.path.join(folder_path, filename)
    if not os.path.exists(path):
        raise ValueError(f"{path} does not exist")
    return True


def get_sub_folders_names(folder_path):
    sub_folders_name = [
        class_name
        for class_name in os.listdir(folder_path)
        if os.path.isdir(os.path.join(folder_path, class_name))
    ]
    return sub_folders_name


def save_pickle(folder_path, obj, filename):
    filepath = os.path.join(folder_path, filename)
    with open(filepath, "wb") as f:
        pickle.dump(obj, f)


def load_pickle(folder_path, filename):
    filepath = os.path.join(folder_path, filename)
    with open(filepath, "rb") as f:
        obj = pickle.load(f)
    return obj


def lower_data(df):
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].apply(lambda x: x.lower() if isinstance(x, str) else x)


def drop_columns(X, col_drop):
    col_drop = list(col_drop.keys())
    if col_drop:
        X.drop(columns=col_drop, inplace=True, errors="ignore")


def is_valid_image(image_path):
    valid_extension = (".jpg", ".png", ".jpeg")

    if not os.path.isfile(image_path):
        return False
    if image_path.endswith(valid_extension):
        try:
            with Image.open(image_path) as img:
                img.verify()
            return True
        except Exception:
            return False
    return False


def collect_function(batch):
    images = [item[0] for item in batch]
    labels = [item[1] for item in batch]
    images_stack = torch.stack(images, dim=0)
    if labels[0] is None:
        return images_stack
    labels = torch.tensor(labels, dtype=torch.long)
    return images_stack, labels


def collect_function_segmentation(batch):
    images = [item[0] for item in batch]
    masks = [item[1] for item in batch]
    images_stack = torch.stack(images, dim=0)
    if masks[0] is None:
        return images_stack
    masks_stack = torch.stack([mask for mask in masks], dim=0)
    return images_stack, masks_stack

In [8]:
class PreprocessingBase:
    def __init__(self, folder_path):
        self.folder_path = folder_path
        if folder_path is None:
            raise ValueError("folder_path cannot be None")

    def common_preprocessing(self):
        raise NotImplementedError("Must implement common_preprocessing method.")

In [9]:
from sklearn.model_selection import train_test_split


class ImageTrainingPreprocessing(PreprocessingBase):
    def __init__(
        self,
        training_folder,
        folder_path,
        validation_folder,
        split_training,
        val_size,
        random_state,
        images_size,
        augment=True,
        batch_size=16,
        augmentation_probability=0.5,
        horizontal_flip=True,
        vertical_flip=True,
        rotation=True,
        rotation_angle=30,
        brightness=True,
        brightness_factors=(0.8, 1.2),
        contrast=True,
        contrast_factors=(0.8, 1.2),
    ):
        super().__init__(folder_path)
        self.training_folder = training_folder
        self.validation_folder = validation_folder
        self.split_training = split_training
        self.val_size = val_size
        self.random_state = random_state
        self.image_size = images_size
        self.augment = augment
        self.batch_size = batch_size
        self.horizontal_flip = horizontal_flip
        self.vertical_flip = vertical_flip
        self.augmentation_probability = augmentation_probability
        self.rotation = rotation
        self.rotation_angle = rotation_angle
        self.brightness = brightness
        self.brightness_factors = brightness_factors
        self.contrast = contrast
        self.contrast_factors = contrast_factors
        self.report_data = {}
        if self.training_folder is None:
            raise ValueError("Training folder must be not null")
        check_path_exists(self.training_folder, "")
        if self.validation_folder is not None:
            check_path_exists(self.validation_folder, "")
            if self.split_training:
                raise ValueError(
                    "The validation folder not none so there is no need "
                    "for splitting data again to training and validation"
                )
            if self.training_folder == self.validation_folder:
                raise ValueError(
                    "The validation folder and training folder must be different"
                )

        if (not (isinstance(self.val_size, (float)))) or (
            not (0 < self.val_size <= 0.5)
        ):
            raise ValueError("Test size is invalid it must be less than 0.5")
        if (self.random_state is not None) and not (
            isinstance(self.random_state, int)
        ):
            raise ValueError("Random state is invalid it must be integer or None")
        if not isinstance(self.image_size, tuple):
            raise ValueError("Image size must be tuple")
        if not isinstance(self.image_size[0], int) or not isinstance(
            self.image_size[1], int
        ):
            raise ValueError("Image size is not integer")
        if not (32 <= self.image_size[0] <= 1024) or not (
            32 <= self.image_size[1] <= 1024
        ):
            raise ValueError(
                "Image size is must be greater than or equal 32 and "
                "less than or equal 1024"
            )
        if not isinstance(self.image_size, tuple):
            raise ValueError("Image size must be tuple")
        if not isinstance(self.image_size[0], int) or not isinstance(
            self.image_size[1], int
        ):
            raise ValueError("Image size is not integer")
        if not (32 <= self.image_size[0] <= 1024) or not (
            32 <= self.image_size[1] <= 1024
        ):
            raise ValueError(
                "Image size is must be greater than or equal 32 and "
                "less than or equal 1024"
            )
        if not isinstance(self.batch_size, int):
            raise ValueError("batch_size must be a integer")

        if self.batch_size <= 0:
            raise ValueError("batch_size must be a positive value")

        if not isinstance(self.augment, bool):
            raise ValueError("augment must be a boolean")

        if not isinstance(self.horizontal_flip, bool):
            raise ValueError("horizontal_flip must be a boolean")

        if not isinstance(self.vertical_flip, bool):
            raise ValueError("vertical_flip must be a boolean")

        if not isinstance(self.rotation, bool):
            raise ValueError("rotation must be a boolean")

        if not isinstance(self.brightness, bool):
            raise ValueError("brightness must be a boolean")

        if not isinstance(self.contrast, bool):
            raise ValueError("contrast must be a boolean")
        if not isinstance(self.augmentation_probability, (int, float)) or not (
            0 <= self.augmentation_probability <= 1
        ):
            raise ValueError("augmentation_probability must be between [0,1]")
        if (
            not isinstance(self.rotation_angle, (int, float))
            or self.rotation_angle < 0
        ):
            raise ValueError("rotation_angle must be positive integer or float")

        if (
            not isinstance(self.brightness_factors, tuple)
            or len(self.brightness_factors) != 2
            or not all(isinstance(x, (int, float)) for x in self.brightness_factors)
        ):
            raise ValueError(
                "brightness_factors must be tuple of two items float or integers"
            )
        if (
            not isinstance(self.contrast_factors, tuple)
            or len(self.contrast_factors) != 2
            or not all(isinstance(x, (int, float)) for x in self.contrast_factors)
        ):
            raise ValueError(
                "contrast_factors must be tuple of two items float or integers"
            )
        os.makedirs(self.folder_path, exist_ok=True)

    def get_sample_random(self, data_type, images_path, labels, num_samples=5):
        df = (
            pd.DataFrame(
                {
                    f"{data_type}_paths": images_path,
                    f"{data_type}_labels": labels,
                }
            )
            .sample(frac=1, random_state=self.random_state)
            .reset_index(drop=True)
        ).head(num_samples)
        return df

    def splitting(
        self,
        training_folder_images_paths,
        validation_folder_images_paths,
        training_folder_images_labels,
        validation_folder_images_labels,
    ):
        if self.split_training:
            (
                self.training_paths,
                self.validation_paths,
                self.training_labels,
                self.validation_labels,
            ) = train_test_split(
                training_folder_images_paths,
                training_folder_images_labels,
                test_size=self.val_size,
                random_state=self.random_state,
            )
            self.report_data["split_data"] = {
                "validation_size": self.val_size,
                "training_data_size": len(self.training_paths),
                "random_training_sample": self.get_sample_random(
                    "training", self.training_paths, self.training_labels
                ),
                "validation_data_size": len(self.validation_paths),
                "random_validation_sample": self.get_sample_random(
                    "validation", self.validation_paths, self.validation_labels
                ),
            }
        else:
            (
                self.training_paths,
                self.validation_paths,
                self.training_labels,
                self.validation_labels,
            ) = (
                training_folder_images_paths,
                validation_folder_images_paths,
                training_folder_images_labels,
                validation_folder_images_labels,
            )

In [10]:
import random

from PIL import ImageEnhance
from torch.utils.data import Dataset


class ImageDataset(Dataset):
    def __init__(
        self,
        image_paths,
        labels=None,
        image_size=(224, 224),
        augment=True,
        augmentation_probability=0.5,
        horizontal_flip=True,
        vertical_flip=True,
        rotation=True,
        rotation_angle=30,
        brightness=True,
        brightness_factors=(0.8, 1.2),
        contrast=True,
        contrast_factors=(0.8, 1.2),
    ):
        self.image_paths = image_paths
        self.labels = labels
        self.image_size = image_size
        self.augment = augment
        self.horizontal_flip = horizontal_flip
        self.vertical_flip = vertical_flip
        self.augmentation_probability = augmentation_probability
        self.rotation = rotation
        self.rotation_angle = rotation_angle
        self.brightness = brightness
        self.brightness_factors = brightness_factors
        self.contrast = contrast
        self.contrast_factors = contrast_factors

    def __len__(self):
        return len(self.image_paths)

    def random_brightness(self, img):
        if self.brightness and random.random() < self.augmentation_probability:
            random_factor = random.uniform(*self.brightness_factors)
            return ImageEnhance.Brightness(img).enhance(random_factor)
        return img

    def random_contrast(self, img):
        if self.contrast and random.random() < self.augmentation_probability:
            random_factor = random.uniform(*self.contrast_factors)
            return ImageEnhance.Contrast(img).enhance(random_factor)
        return img

In [11]:
class ClassificationImageDataset(ImageDataset):
    def __init__(
        self,
        image_paths,
        labels=None,
        image_size=(224, 224),
        augment=False,
        augmentation_probability=0.5,
        horizontal_flip=False,
        vertical_flip=False,
        rotation=False,
        rotation_angle=30,
        brightness=False,
        brightness_factors=(0.8, 1.2),
        contrast=False,
        contrast_factors=(0.8, 1.2),
    ):
        super().__init__(
            image_paths,
            labels,
            image_size,
            augment,
            augmentation_probability,
            horizontal_flip,
            vertical_flip,
            rotation,
            rotation_angle,
            brightness,
            brightness_factors,
            contrast,
            contrast_factors,
        )

    def load_image(self, index):
        path = self.image_paths[index]
        img = Image.open(path).convert("RGB")
        img = img.resize(self.image_size)
        if self.augment:
            img = self.augmentation(img)
        img_array = np.array(img, dtype=np.float32) / 255.0
        img_transpose = np.transpose(img_array, (2, 0, 1))
        img_tensor = torch.tensor(img_transpose, dtype=torch.float32)
        return img_tensor

    def random_horizontal_flip(self, img):
        if self.horizontal_flip and random.random() < self.augmentation_probability:
            return img.transpose(Image.FLIP_LEFT_RIGHT)
        return img

    def random_vertical_flip(self, img):
        if self.vertical_flip and random.random() < self.augmentation_probability:
            return img.transpose(Image.FLIP_TOP_BOTTOM)
        return img

    def random_rotation(self, img):
        if self.rotation and random.random() < self.augmentation_probability:
            random_angle = random.uniform(
                -1 * self.rotation_angle, self.rotation_angle
            )
            return img.rotate(random_angle)
        return img

    def augmentation(self, img):
        img = self.random_horizontal_flip(img)
        img = self.random_vertical_flip(img)
        img = self.random_rotation(img)
        img = self.random_brightness(img)
        img = self.random_contrast(img)
        return img

    def __getitem__(self, index):
        img_tensor = self.load_image(index)
        label_tensor = None
        if self.labels is not None:
            label_tensor = torch.tensor(self.labels[index], dtype=torch.long)
        return img_tensor, label_tensor

In [12]:
from torch.utils.data import DataLoader


class ClassificationImageTrainingPreprocessing(ImageTrainingPreprocessing):
    def __init__(
        self,
        training_folder_images,
        folder_path,
        validation_folder_images=None,
        split_training=False,
        val_size=0.2,
        random_state=42,
        images_size=(224, 224),
        augment=True,
        batch_size=16,
        augmentation_probability=0.5,
        horizontal_flip=True,
        vertical_flip=True,
        rotation=True,
        rotation_angle=30,
        brightness=True,
        brightness_factors=(0.8, 1.2),
        contrast=True,
        contrast_factors=(0.8, 1.2),
    ):
        super().__init__(
            training_folder_images,
            folder_path,
            validation_folder_images,
            split_training,
            val_size,
            random_state,
            images_size,
            augment,
            batch_size,
            augmentation_probability,
            horizontal_flip,
            vertical_flip,
            rotation,
            rotation_angle,
            brightness,
            brightness_factors,
            contrast,
            contrast_factors,
        )
        self.training_class = get_sub_folders_names(self.training_folder)
        self.validation_class = None
        self.training_folder_invalid_images = []
        self.training_folder_invalid_images_labels = []
        self.validation_folder_invalid_images = []
        self.validation_folder_invalid_images_labels = []
        if len(self.training_class) == 0:
            raise ValueError("Training Folder dosen't have any folder inside it ")
        if self.validation_folder is not None:
            self.validation_class = get_sub_folders_names(self.validation_folder)
            if len(self.validation_class) == 0:
                raise ValueError(
                    "Validation Folder dosen't have any folder inside it "
                )
            for class_name in self.validation_class:
                if class_name not in self.training_class:
                    raise ValueError(
                        f"This category {class_name} not in the training "
                        f"categories which are {self.training_class}"
                    )
        (
            self.validation_folder_images_paths,
            self.validation_folder_images_labels,
        ) = (
            None,
            None,
        )
        data_info = {"image_size": self.image_size}
        save_pickle(self.folder_path, data_info, "data_info.pkl")

    def get_classes_mapping(self):
        self.class2label_mapping = {}
        self.label2class_mapping = {}
        for idx, class_name in enumerate(sorted(self.training_class)):
            self.class2label_mapping[class_name] = idx
            self.label2class_mapping[idx] = class_name
        save_pickle(
            self.folder_path,
            self.class2label_mapping,
            "class2label_mapping.pkl",
        )
        save_pickle(
            self.folder_path,
            self.label2class_mapping,
            "label2class_mapping.pkl",
        )

    def data_preparing(
        self, folder_path, invalid_list_paths, invalid_list_labels, classes_name
    ):
        paths = []
        labels = []
        for class_name in classes_name:
            sub_folder_path = os.path.join(folder_path, class_name)
            for path in os.listdir(sub_folder_path):
                path = os.path.join(sub_folder_path, path)
                mapping = self.class2label_mapping[class_name]
                if is_valid_image(path):
                    paths.append(path)
                    labels.append(mapping)
                else:
                    invalid_list_paths.append(path)
                    invalid_list_labels.append(mapping)
        if len(paths) == 0:
            raise ValueError("There is no valid path")
        return paths, labels

    def data_overview(self):
        train_df = self.get_sample_random(
            "training_folder",
            self.training_folder_images_paths,
            self.training_folder_images_labels,
        )
        self.report_data["data_overview"] = {}
        self.report_data["data_overview"]["training_folder"] = {
            "classes": self.training_class,
            "images_len": len(self.training_folder_images_paths),
            "random_sample": train_df.head(),
            "invalid_len": len(self.training_folder_invalid_images),
            "invalid_images": self.training_folder_invalid_images,
            "mapping": self.class2label_mapping,
        }

        if self.validation_folder is not None:
            val_df = self.get_sample_random(
                "validation_folder",
                self.validation_folder_images_paths,
                self.validation_folder_images_labels,
            )
            self.report_data["data_overview"]["validation_folder"] = {
                "classes": self.validation_class,
                "images_len": len(self.validation_folder_images_paths),
                "random_sample": val_df.head(),
                "invalid_len": len(self.validation_folder_invalid_images),
                "invalid_images": self.validation_folder_invalid_images,
            }

    def make_graphs_label_summary(self):
        ImageLabelClassification(
            self.training_folder_images_labels,
            self.label2class_mapping,
            self.training_folder_invalid_images_labels,
            self.folder_path,
            "Training Folder Labels Summary ",
            "training_folder_labels_summary",
        ).build_graph()
        self.report_data["graphs"]["images_name"].append(
            "training_folder_labels_summary"
        )
        if self.validation_folder is not None:
            ImageLabelClassification(
                self.validation_folder_images_labels,
                self.label2class_mapping,
                self.validation_folder_invalid_images_labels,
                self.folder_path,
                "Validation Folder Labels Summary ",
                "validation_folder_labels_summary",
            ).build_graph()
            self.report_data["graphs"]["images_name"].append(
                "validation_folder_labels_summary"
            )
        if self.split_training:
            ImageLabelClassification(
                self.training_labels,
                self.label2class_mapping,
                None,
                self.folder_path,
                "Training Data After Splitting Labels Summary ",
                "training_after_splitting_labels_summary",
            ).build_graph()
            self.report_data["graphs"]["images_name"].append(
                "training_after_splitting_labels_summary"
            )
            ImageLabelClassification(
                self.validation_labels,
                self.label2class_mapping,
                None,
                self.folder_path,
                "Validation Data After Splitting Labels Summary ",
                "validation_after_splitting_labels_summary",
            ).build_graph()
            self.report_data["graphs"]["images_name"].append(
                "validation_after_splitting_labels_summary"
            )

    def make_garphs_sample(self):
        DisplaySampleImagesClassification(
            self.training_folder_images_paths,
            self.training_folder_images_labels,
            self.label2class_mapping,
            self.folder_path,
            title=" Random Samples of Training Folder",
            file_name="training_folder_random_samples",
        ).build_graph()
        self.report_data["graphs"]["images_name"].append(
            "training_folder_random_samples"
        )
        if self.validation_folder is not None:
            DisplaySampleImagesClassification(
                self.validation_folder_images_paths,
                self.validation_folder_images_labels,
                self.label2class_mapping,
                self.folder_path,
                title="Random Samples of Validation Folder",
                file_name="validation_folder_random_samples",
            ).build_graph()
            self.report_data["graphs"]["images_name"].append(
                "validation_folder_random_samples"
            )
        if self.split_training:
            DisplaySampleImagesClassification(
                self.training_paths,
                self.training_labels,
                self.label2class_mapping,
                self.folder_path,
                title="Samples of Training Data After Splitting",
                file_name="training_after_splitting_random_samples",
            ).build_graph()
            self.report_data["graphs"]["images_name"].append(
                "training_after_splitting_random_samples"
            )
            DisplaySampleImagesClassification(
                self.validation_paths,
                self.validation_labels,
                self.label2class_mapping,
                self.folder_path,
                title="Samples of Validation Data After Splitting",
                file_name="validation_after_splitting_random_samples",
            ).build_graph()
            self.report_data["graphs"]["images_name"].append(
                "validation_after_splitting_random_samples"
            )

    def make_graphs_loader(self):
        training_images, training_labels = next(iter(self.training_loader))
        n_samples = min(5, len(training_images))

        training_images, training_labels = (
            training_images[:n_samples],
            training_labels[:n_samples],
        )
        ClassificationImagesAfterLoader(
            training_images,
            training_labels,
            self.label2class_mapping,
            self.folder_path,
            title="Samples of Training Data After Data Loader",
            file_name="training_after_data_loader_samples",
        ).build_graph()
        self.report_data["graphs"]["images_name"].append(
            "training_after_data_loader_samples"
        )
        if self.validation_loader is not None:
            validation_images, validation_labels = next(iter(self.validation_loader))
            n_samples = min(5, len(validation_images))
            validation_images, validation_labels = (
                validation_images[:n_samples],
                validation_labels[:n_samples],
            )
            ClassificationImagesAfterLoader(
                validation_images,
                validation_labels,
                self.label2class_mapping,
                self.folder_path,
                title="Samples of Validation Data After Data Loader",
                file_name="validation_after_data_loader_samples",
            ).build_graph()
            self.report_data["graphs"]["images_name"].append(
                "validation_after_data_loader_samples"
            )

    def make_graphs(self):
        self.report_data["graphs"] = {
            "folder_path": self.folder_path,
            "images_name": [],
        }
        self.make_graphs_label_summary()
        self.make_garphs_sample()
        self.make_graphs_loader()

    def get_loaders(self):
        training_dataset = ClassificationImageDataset(
            self.training_paths,
            self.training_labels,
            self.image_size,
            self.augment,
            self.augmentation_probability,
            self.horizontal_flip,
            self.vertical_flip,
            self.rotation,
            self.rotation_angle,
            self.brightness,
            self.brightness_factors,
            self.contrast,
            self.contrast_factors,
        )
        self.training_loader = DataLoader(
            training_dataset, batch_size=self.batch_size, shuffle=True
        )
        self.validation_loader = None
        if self.validation_paths is not None:
            validation_dataset = ClassificationImageDataset(
                self.validation_paths,
                self.validation_labels,
                self.image_size,
                False,
                self.augmentation_probability,
                self.horizontal_flip,
                self.vertical_flip,
                self.rotation,
                self.rotation_angle,
                self.brightness,
                self.brightness_factors,
                self.contrast,
                self.contrast_factors,
            )
            self.validation_loader = DataLoader(
                validation_dataset, batch_size=self.batch_size, shuffle=False
            )

    def common_preprocessing(self):
        self.get_classes_mapping()
        (
            self.training_folder_images_paths,
            self.training_folder_images_labels,
        ) = self.data_preparing(
            self.training_folder,
            self.training_folder_invalid_images,
            self.training_folder_invalid_images_labels,
            self.training_class,
        )

        if self.validation_folder is not None:
            (
                self.validation_folder_images_paths,
                self.validation_folder_images_labels,
            ) = self.data_preparing(
                self.validation_folder_images,
                self.validation_folder_invalid_images,
                self.validation_folder_invalid_images_labels,
                self.validation_class,
            )
        self.data_overview()
        self.splitting(
            self.training_folder_images_paths,
            self.validation_folder_images_paths,
            self.training_folder_images_labels,
            self.validation_folder_images_labels,
        )
        self.get_loaders()
        self.make_graphs()

        report = ImagePreprocessingReport(self.folder_path, self.report_data)
        report.execute()
        return self.training_loader, self.validation_loader

In [13]:
class ClassificationImageTestingPreprocessing(PreprocessingBase):
    def __init__(
        self,
        image_paths,
        image_class_names=None,
        folder_path=None,
        batch_size=4,
    ):
        super().__init__(folder_path)
        self.image_paths = image_paths
        self.image_class_names = image_class_names
        self.batch_size = batch_size
        self.valid_images = []
        self.valid_images_class_names = []
        self.invalid_images = []
        if self.image_paths is None:
            raise ValueError("Image paths must be list of paths not none")

        if not isinstance(self.image_paths, list):
            raise ValueError("Image paths must be list of paths")
        if len(self.image_paths) == 0:
            raise ValueError("Image paths is empty list")
        if self.image_class_names is not None and not isinstance(
            self.image_class_names, list
        ):
            raise ValueError("Class names must be list of class names")
        if self.image_class_names is not None and len(self.image_class_names) != len(
            self.image_paths
        ):
            raise ValueError("Image paths length must equal class names length")
        for i, path in enumerate(self.image_paths):
            if is_valid_image(path):
                self.valid_images.append(path)
                if self.image_class_names is not None:
                    self.valid_images_class_names.append(self.image_class_names[i])
            else:
                self.invalid_images.append(path)
        if len(self.valid_images) == 0:
            raise ValueError("There is no valid image exists")

        check_path_exists(self.folder_path, "data_info.pkl")
        self.image_size = load_pickle(self.folder_path, "data_info.pkl")[
            "image_size"
        ]
        check_path_exists(self.folder_path, "class2label_mapping.pkl")
        self.class2label_mapping = load_pickle(
            self.folder_path, "class2label_mapping.pkl"
        )

    def common_preprocessing(self):
        labels = None
        if self.image_class_names is not None:
            labels = []
            for class_name in self.valid_images_class_names:
                if class_name not in self.class2label_mapping:
                    raise ValueError("this class name not in the training class")
                labels.append(self.class2label_mapping[class_name])

        dataset = ClassificationImageDataset(
            self.valid_images, labels, self.image_size, augment=False
        )
        testing_loader = DataLoader(
            dataset,
            batch_size=self.batch_size,
            shuffle=False,
            collate_fn=collect_function,
        )
        return testing_loader, self.invalid_images

In [16]:
training_preprocessor = ClassificationImageTrainingPreprocessing(
    training_folder_images="/kaggle/input/datasets/bhavikjikadara/dog-and-cat-classification-dataset/PetImages",
    folder_path="/kaggle/working/PetImagesReport",
    validation_folder_images=None,
    split_training=True,
    val_size=0.2,
    random_state=23,
    images_size=(224, 224),
    augment=True,
    horizontal_flip=True,
    vertical_flip=True,
    rotation=True,
    brightness=True,
    contrast=True,
)
training_loader, validation_loader = training_preprocessor.common_preprocessing()

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


In [14]:
def training_func(
    model,
    criterion,
    optimizer,
    epochs,
    training_loader,
    validation_loader,
    model_name,
):
    best_acc = 0
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in training_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_accuracy = 100 * train_correct / train_total

        model.eval()
        val_correct = 0
        val_total = 0
        val_loss = 0.0

        with torch.no_grad():
            for images, labels in validation_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()

                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_accuracy = 100 * val_correct / val_total
        if val_accuracy > best_acc:
            best_acc = val_accuracy
            torch.save(
                model.state_dict(),
                f"/kaggle/working/{model_name}_best_model.pth",
            )
            print(f"saved model with val accuracy {val_accuracy}")

        print(f"Epoch [{epoch + 1}/{epochs}]")
        print(
            f"Train Loss: {train_loss / len(training_loader):.4f} | Train Acc: {train_accuracy:.2f}%"
        )
        print(
            f"Val   Loss: {val_loss / len(validation_loader):.4f} | Val   Acc: {val_accuracy:.2f}%"
        )
        print("-" * 20)

In [17]:
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
classes = len(training_preprocessor.class2label_mapping)

model = models.resnet18(weights="IMAGENET1K_V1")
model.fc = nn.Linear(model.fc.in_features, classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10

training_func(
    model,
    criterion,
    optimizer,
    epochs,
    training_loader,
    validation_loader,
    "classification_tuning",
)

Using device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 170MB/s]
/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


saved model with val accuracy 87.92
Epoch [1/10]
Train Loss: 0.3327 | Train Acc: 86.02%
Val   Loss: 0.2766 | Val   Acc: 87.92%
--------------------
saved model with val accuracy 92.24
Epoch [2/10]
Train Loss: 0.2449 | Train Acc: 89.76%
Val   Loss: 0.1896 | Val   Acc: 92.24%
--------------------
Epoch [3/10]
Train Loss: 0.2149 | Train Acc: 91.18%
Val   Loss: 0.2887 | Val   Acc: 88.28%
--------------------
Epoch [4/10]
Train Loss: 0.1926 | Train Acc: 92.11%
Val   Loss: 0.1759 | Val   Acc: 91.86%
--------------------
saved model with val accuracy 94.4
Epoch [5/10]
Train Loss: 0.1775 | Train Acc: 92.79%
Val   Loss: 0.1333 | Val   Acc: 94.40%
--------------------
saved model with val accuracy 95.46
Epoch [6/10]
Train Loss: 0.1625 | Train Acc: 93.35%
Val   Loss: 0.1111 | Val   Acc: 95.46%
--------------------
Epoch [7/10]
Train Loss: 0.1514 | Train Acc: 93.94%
Val   Loss: 0.1328 | Val   Acc: 94.40%
--------------------
Epoch [8/10]
Train Loss: 0.1369 | Train Acc: 94.35%
Val   Loss: 0.1148 | 

In [18]:
!zip -r PetImagesReport.zip /kaggle/working/PetImagesReport

  adding: kaggle/working/PetImagesReport/ (stored 0%)
  adding: kaggle/working/PetImagesReport/report.html (deflated 82%)
  adding: kaggle/working/PetImagesReport/label2class_mapping.pkl (deflated 6%)
  adding: kaggle/working/PetImagesReport/class2label_mapping.pkl (deflated 6%)
  adding: kaggle/working/PetImagesReport/data_info.pkl (deflated 6%)
  adding: kaggle/working/PetImagesReport/graphs/ (stored 0%)
  adding: kaggle/working/PetImagesReport/graphs/validation_after_splitting_labels_summary.png (deflated 12%)
  adding: kaggle/working/PetImagesReport/graphs/training_after_splitting_labels_summary.png (deflated 13%)
  adding: kaggle/working/PetImagesReport/graphs/training_after_splitting_random_samples.png (deflated 0%)
  adding: kaggle/working/PetImagesReport/graphs/validation_after_data_loader_samples.png (deflated 0%)
  adding: kaggle/working/PetImagesReport/graphs/validation_after_splitting_random_samples.png (deflated 0%)
  adding: kaggle/working/PetImagesReport/graphs/training_

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models


class CNN_Model(nn.Module):
    def __init__(self, classes):
        super(CNN_Model, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 28 * 28, 512)
        self.fc2 = nn.Linear(512, classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
classes = len(training_preprocessor.class2label_mapping)

model = CNN_Model(classes)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 15

training_func(
    model,
    criterion,
    optimizer,
    epochs,
    training_loader,
    validation_loader,
    "classification_with_agument",
)

Using device: cuda


/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


saved model with val accuracy 57.16
Epoch [1/15]
Train Loss: 0.6930 | Train Acc: 53.55%
Val   Loss: 0.6773 | Val   Acc: 57.16%
--------------------
saved model with val accuracy 60.9
Epoch [2/15]
Train Loss: 0.6810 | Train Acc: 56.54%
Val   Loss: 0.6656 | Val   Acc: 60.90%
--------------------
saved model with val accuracy 63.22
Epoch [3/15]
Train Loss: 0.6711 | Train Acc: 58.40%
Val   Loss: 0.6451 | Val   Acc: 63.22%
--------------------
saved model with val accuracy 67.32
Epoch [4/15]
Train Loss: 0.6277 | Train Acc: 65.19%
Val   Loss: 0.6059 | Val   Acc: 67.32%
--------------------
saved model with val accuracy 70.88
Epoch [5/15]
Train Loss: 0.5912 | Train Acc: 68.65%
Val   Loss: 0.5819 | Val   Acc: 70.88%
--------------------
saved model with val accuracy 72.08
Epoch [6/15]
Train Loss: 0.5709 | Train Acc: 70.35%
Val   Loss: 0.5564 | Val   Acc: 72.08%
--------------------
saved model with val accuracy 74.08
Epoch [7/15]
Train Loss: 0.5522 | Train Acc: 71.85%
Val   Loss: 0.5292 | Val 

In [21]:
!zip -r PetImagesReport.zip /kaggle/working/PetImagesReport

  adding: kaggle/working/PetImagesReport/ (stored 0%)
  adding: kaggle/working/PetImagesReport/data_info.pkl (deflated 6%)
  adding: kaggle/working/PetImagesReport/class2label_mapping.pkl (deflated 6%)
  adding: kaggle/working/PetImagesReport/label2class_mapping.pkl (deflated 6%)
  adding: kaggle/working/PetImagesReport/report.html (deflated 82%)
  adding: kaggle/working/PetImagesReport/graphs/ (stored 0%)
  adding: kaggle/working/PetImagesReport/graphs/training_after_data_loader_samples.png (deflated 0%)
  adding: kaggle/working/PetImagesReport/graphs/training_after_splitting_labels_summary.png (deflated 13%)
  adding: kaggle/working/PetImagesReport/graphs/validation_after_data_loader_samples.png (deflated 0%)
  adding: kaggle/working/PetImagesReport/graphs/validation_after_splitting_labels_summary.png (deflated 12%)
  adding: kaggle/working/PetImagesReport/graphs/validation_after_splitting_random_samples.png (deflated 0%)
  adding: kaggle/working/PetImagesReport/graphs/training_folde

In [22]:
!zip -r model.zip /kaggle/working/classification_with_agument_best_model.pth

  adding: kaggle/working/classification_with_agument_best_model.pth (deflated 8%)


In [16]:
training_preprocessor = ClassificationImageTrainingPreprocessing(
    training_folder_images="/kaggle/input/datasets/bhavikjikadara/dog-and-cat-classification-dataset/PetImages",
    folder_path="PetImagesReportWithoutAugment",
    validation_folder_images=None,
    split_training=True,
    val_size=0.2,
    random_state=23,
    images_size=(224, 224),
    augment=False,
)
training_loader, validation_loader = training_preprocessor.common_preprocessing()

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


In [17]:
!zip -r PetImagesReportWithoutAugment.zip /kaggle/working/PetImagesReportWithoutAugment

  adding: kaggle/working/PetImagesReportWithoutAugment/ (stored 0%)
  adding: kaggle/working/PetImagesReportWithoutAugment/data_info.pkl (deflated 6%)
  adding: kaggle/working/PetImagesReportWithoutAugment/report.html (deflated 82%)
  adding: kaggle/working/PetImagesReportWithoutAugment/label2class_mapping.pkl (deflated 6%)
  adding: kaggle/working/PetImagesReportWithoutAugment/class2label_mapping.pkl (deflated 6%)
  adding: kaggle/working/PetImagesReportWithoutAugment/graphs/ (stored 0%)
  adding: kaggle/working/PetImagesReportWithoutAugment/graphs/validation_after_splitting_random_samples.png (deflated 0%)
  adding: kaggle/working/PetImagesReportWithoutAugment/graphs/training_after_splitting_random_samples.png (deflated 0%)
  adding: kaggle/working/PetImagesReportWithoutAugment/graphs/validation_after_splitting_labels_summary.png (deflated 12%)
  adding: kaggle/working/PetImagesReportWithoutAugment/graphs/validation_after_data_loader_samples.png (deflated 0%)
  adding: kaggle/working

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
classes = len(training_preprocessor.class2label_mapping)

model = CNN_Model(classes)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 15

training_func(
    model,
    criterion,
    optimizer,
    epochs,
    training_loader,
    validation_loader,
    "classification_without_agument",
)

Using device: cuda


/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


saved model with val accuracy 50.12
Epoch [1/15]
Train Loss: 0.7009 | Train Acc: 49.28%
Val   Loss: 0.6931 | Val   Acc: 50.12%
--------------------
Epoch [2/15]
Train Loss: 0.6932 | Train Acc: 50.24%
Val   Loss: 0.6932 | Val   Acc: 49.88%
--------------------
Epoch [3/15]
Train Loss: 0.6933 | Train Acc: 49.79%
Val   Loss: 0.6932 | Val   Acc: 49.88%
--------------------
Epoch [4/15]
Train Loss: 0.6932 | Train Acc: 49.53%
Val   Loss: 0.6931 | Val   Acc: 50.12%
--------------------
saved model with val accuracy 65.48
Epoch [5/15]
Train Loss: 0.6858 | Train Acc: 53.95%
Val   Loss: 0.6325 | Val   Acc: 65.48%
--------------------
saved model with val accuracy 71.88
Epoch [6/15]
Train Loss: 0.5983 | Train Acc: 67.85%
Val   Loss: 0.5433 | Val   Acc: 71.88%
--------------------
saved model with val accuracy 76.2
Epoch [7/15]
Train Loss: 0.4994 | Train Acc: 75.92%
Val   Loss: 0.5020 | Val   Acc: 76.20%
--------------------
Epoch [8/15]
Train Loss: 0.4285 | Train Acc: 80.33%
Val   Loss: 0.4990 | 

In [21]:
testing_loader, invalid_path = ClassificationImageTestingPreprocessing(
    [
        "/kaggle/input/datasets/bhavikjikadara/dog-and-cat-classification-dataset/PetImages/Cat/10022.jpg",
        "/kaggle/input/datasets/bhavikjikadara/dog-and-cat-classification-dataset/PetImages/Dog/10000.jpg",
        "/kaggle/input/datasets/nesmaosamaahmed/testing-cat2/Screenshot (75).png",
    ],
    image_class_names=["Cat", "Dog", "Cat"],
    folder_path="/kaggle/working/PetImagesReportWithoutAugment",
).common_preprocessing()
print(invalid_path)

[]


In [22]:
model.eval()
val_loss = 0
with torch.no_grad():
    for images, labels in testing_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        val_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        print("predicted", predicted)
        print("corrected", labels)

predicted tensor([0, 1, 0], device='cuda:0')
corrected tensor([0, 1, 0], device='cuda:0')
